# Theme

This project analyzes a stock by combining its recent price trend with the sentiment of related news, then simulates whether the user should Buy, Hold, or Sell—displaying an explanation and price chart for easy decision-making.

# Install

yfinance → Fetches real-time stock prices and historical market data.

transformers → Loads the sentiment model used to classify news as positive, neutral, or negative.

torch → Runs the machine learning model efficiently.

gradio → Builds the user interface for interactive stock analysis.

matplotlib → Plots stock price charts with moving averages.

In [ ]:
!pip install yfinance transformers torch gradio matplotlib --quiet

Imports & Model Setup Cell  Loads yfinance, transformers, torch, and Gradio, and prepares the environment for stock analysis and sentiment modeling.

In [ ]:
!pip install yfinance transformers torch gradio matplotlib -q

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import gradio as gr

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

plt.style.use("seaborn-v0_8")


1- Yahoo Finance = A website (and system) that has stock market data.

2- The stock must exist on Yahoo Finance.

# Load FAST LLM Sentiment Model

Sentiment Model Loading Cell → Initializes the MiniLM-based model to classify news headlines as positive, neutral, or negative.

In [ ]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The MiniLM-based sentiment model is:

✔ Lightweight (70 MB)
✔ Loads instantly
✔ Works smoothly in Google Colab
✔ Ideal for continuous user interaction

This makes your project demo fast and professional.

# MiniLM Sentiment Analyzer

analyze_sentiment() Function Cell → Converts a news headline into a sentiment label and confidence score using the sentiment model.

In [ ]:
def analyze_sentiment(text):
    """
    Converts news/headline → positive / neutral / negative using MiniLM.
    """
    if not text or text.strip() == "":
        return "neutral", 0.0

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)

    with torch.no_grad():
        outputs = sentiment_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).numpy()[0]

    stars = int(np.argmax(probs)) + 1

    if stars <= 2:
        label = "negative"
    elif stars == 3:
        label = "neutral"
    else:
        label = "positive"

    score = float(probs[stars - 1])
    return label, score


# Helper: Get Stock Data

get_stock_data() Function Cell → Downloads daily stock prices from Yahoo Finance and calculates returns and moving averages.

In [ ]:
def get_stock_data(ticker):
    df = yf.download(ticker, period="1mo", interval="1d", progress=False)
    if df.empty:
        raise ValueError("No data found for this stock.")

    df = df[["Close"]].copy()
    df["return"] = df["Close"].pct_change().fillna(0.0)
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df.dropna(inplace=True)
    return df


analyze_price_trend() Function Cell → Determines if the stock is in an uptrend, downtrend, or sideways based on recent price movement.

In [ ]:
def analyze_price_trend(df):
    recent = df.tail(5)
    recent_return = float((recent["Close"].iloc[-1] / recent["Close"].iloc[0]) - 1.0)

    last = df.iloc[-1]
    price = float(last["Close"])
    ma5 = float(last["ma_5"])
    ma20 = float(last["ma_20"])

    if price > ma5 and ma5 > ma20 and recent_return > 0.01:
        trend = "uptrend"
    elif price < ma5 and ma5 < ma20 and recent_return < -0.01:
        trend = "downtrend"
    else:
        trend = "sideways"

    return trend, recent_return


decision_engine() Function Cell → Combines trend and sentiment to produce a simulated Buy / Hold / Sell recommendation with a risk level.

In [ ]:
def decision_engine(trend, recent_return, sentiment, sentiment_score):
    score = 0

    if trend == "uptrend":
        score += 2
    elif trend == "downtrend":
        score -= 2

    if sentiment == "positive":
        score += 2
    elif sentiment == "negative":
        score -= 2

    if recent_return > 0.10:
        score -= 1

    if score >= 3:
        return "BUY (Simulated)", "Medium", score
    elif score <= -2:
        return "SELL / AVOID (Simulated)", "High", score
    else:
        return "HOLD / WAIT (Simulated)", "Medium to High", score


plot_price_chart() Function Cell → Generates a price chart with moving averages and embeds it as an image for display in the UI.

In [ ]:
def plot_price_chart(df, ticker):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df.index, df["Close"], label="Close Price")
    ax.plot(df.index, df["ma_5"], label="MA 5", alpha=0.6)
    ax.plot(df.index, df["ma_20"], label="MA 20", alpha=0.6)
    ax.set_title(f"{ticker} - Price Trend")
    ax.legend()
    plt.tight_layout()

    buf = BytesIO()
    plt.savefig(buf, format="png")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode()
    plt.close()

    return f"<img src='data:image/png;base64,{encoded}' style='width:100%; border-radius:12px;'/>"


stock_llm_agent() Function Cell → Acts as the stock analysis brain, processing ticker + news sentiment to output the final decision and explanation.

In [ ]:
def stock_llm_agent(ticker, news_text):
    ticker = ticker.upper().strip()
    if ticker == "":
        return "<b>Please enter a stock ticker.</b>", ""

    try:
        df = get_stock_data(ticker)
    except Exception as e:
        return f"<b>Error fetching stock data:</b> {e}", ""

    trend, recent_ret = analyze_price_trend(df)
    sentiment, sentiment_score = analyze_sentiment(news_text)

    action, risk, score = decision_engine(trend, recent_ret, sentiment, sentiment_score)

    explanation = f"""
    <div style='background:white; padding:15px; border-radius:12px; box-shadow:0 4px 10px rgba(0,0,0,0.1);'>
    <h2 style='color:#1a237e;'>AI Stock Analysis for <b>{ticker}</b></h2>
    <ul>
      <li><b>Price Trend:</b> {trend} (5-day return: {round(recent_ret*100,2)}%)</li>
      <li><b>Sentiment:</b> {sentiment} ({round(sentiment_score*100,1)}% confidence)</li>
      <li><b>Decision (Simulated):</b> {action}</li>
      <li><b>Risk:</b> {risk}</li>
      <li><b>Internal Score:</b> {score}</li>
    </ul>
    <p style='color:#b71c1c;font-size:14px;'>
    ⚠️ Educational simulation only — not financial advice.
    </p>
    </div>
    """

    return explanation, plot_price_chart(df, ticker)


# Beautiful Gradio UI

In [ ]:
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #e3f2fd, #fce4ec);
    font-family: 'Segoe UI', sans-serif;
}
"""

with gr.Blocks(css=custom_css, title="LLM Stock Reasoning Agent") as demo:

    gr.HTML("<h1 style='text-align:center;color:#0d47a1;'>💹 LLM Stock Reasoning Agent</h1>")
    gr.HTML("<p style='text-align:center;color:#4a148c;'>Analyze stock trends + news sentiment to simulate Buy / Hold / Sell.</p>")

    with gr.Row():
        with gr.Column():
            ticker_in = gr.Textbox(label="Stock Ticker", value="AAPL")
            news_in = gr.Textbox(lines=4, label="News / Headline")
            btn = gr.Button("🔍 Analyze Stock", variant="primary")

        with gr.Column():
            explanation_out = gr.HTML()
            chart_out = gr.HTML()

    btn.click(stock_llm_agent, inputs=[ticker_in, news_in], outputs=[explanation_out, chart_out])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b1fdc121a2bcfeab0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


It only does:

recent price trend

recent news sentiment

simulated Buy / Hold / Sell

basic risk score

It does NOT predict future growth

For Indian stocks .NS

RELIANCE.NS
TATAELXIS.NS